# ETL — Tabla Puente Video-Hashtag (`bridge_video_hashtag`)

Este notebook extrae las combinaciones únicas de `(video_id, hashtag)` desde la capa Silver (`tiktok_data_eng.silver.silver_tiktok`),
realiza el cruce con la dimensión de hashtags (`tiktok_data_eng.gold.dim_hashtag`) para obtener la clave subrogada `hashtag_id`
y ejecuta un `MERGE` idempotente en la tabla puente `tiktok_data_eng.gold.bridge_video_hashtag`,
asignando automáticamente la clave subrogada autoincremental `video_hashtag_id`.

In [0]:
%sql
-- Inserción / Actualización idempotente (MERGE) en bridge_video_hashtag utilizando CTEs

WITH pares_silver AS (
    SELECT DISTINCT 
        video_id,
        TRIM(hashtag) AS hashtag
    FROM tiktok_data_eng.silver.silver_tiktok
    WHERE video_id IS NOT NULL AND hashtag IS NOT NULL AND TRIM(hashtag) <> ''
),
pares_con_id AS (
    SELECT 
        p.video_id,
        h.hashtag_id
    FROM pares_silver p
    INNER JOIN tiktok_data_eng.gold.dim_hashtag h
        ON p.hashtag = h.hashtag
)
MERGE INTO tiktok_data_eng.gold.bridge_video_hashtag AS target
USING pares_con_id AS source
ON target.video_id = source.video_id AND target.hashtag_id = source.hashtag_id
WHEN NOT MATCHED THEN INSERT (
    video_id,
    hashtag_id,
    _created_at
) VALUES (
    source.video_id,
    source.hashtag_id,
    CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- Validación de calidad y volumetría en bridge_video_hashtag con ID autoincremental
SELECT 
    COUNT(*) AS total_filas,
    COUNT(DISTINCT video_hashtag_id) AS total_ids_autoincrementales,
    MIN(video_hashtag_id) AS min_id,
    MAX(video_hashtag_id) AS max_id,
    COUNT(DISTINCT video_id) AS total_videos,
    COUNT(DISTINCT hashtag_id) AS total_hashtags,
    SUM(CASE WHEN video_hashtag_id IS NULL OR video_id IS NULL OR hashtag_id IS NULL THEN 1 ELSE 0 END) AS nulos,
    COUNT(*) - COUNT(DISTINCT video_id, hashtag_id) AS duplicados_pares
FROM tiktok_data_eng.gold.bridge_video_hashtag;